# Lending Risk & Borrower Behavior Analysis using Association Rules (Apriori)

# 1. Data Ingestion & Initial Observation

In [2]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

print("Loading dataset...")
df = pd.read_csv('clean_for_apriori.csv')

Loading dataset...


In [3]:
df.head()

,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,sub_grade_A1,sub_grade_A2,sub_grade_A3,...,home_ownership_RENT,last_fico_range_low_bin_Low,last_fico_range_low_bin_Medium,last_fico_range_low_bin_High,fico_range_low_bin_Low,fico_range_low_bin_Medium,fico_range_low_bin_High,installment_bin_Low,installment_bin_Medium,installment_bin_High
0,0,0,1,0,0,0,0,0,0,0,...,0,1,0,0,1,0,0,1,0,0
1,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,1
2,0,1,0,0,0,0,0,0,0,0,...,0,0,1,0,0,1,0,0,1,0
3,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,1
4,0,0,0,0,0,1,0,0,0,0,...,0,0,1,0,0,1,0,1,0,0


In [4]:
df.columns

Index(['grade_A', 'grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F',
       'grade_G', 'sub_grade_A1', 'sub_grade_A2', 'sub_grade_A3',
       'sub_grade_A4', 'sub_grade_A5', 'sub_grade_B1', 'sub_grade_B2',
       'sub_grade_B3', 'sub_grade_B4', 'sub_grade_B5', 'sub_grade_C1',
       'sub_grade_C2', 'sub_grade_C3', 'sub_grade_C4', 'sub_grade_C5',
       'sub_grade_D1', 'sub_grade_D2', 'sub_grade_D3', 'sub_grade_D4',
       'sub_grade_D5', 'sub_grade_E1', 'sub_grade_E2', 'sub_grade_E3',
       'sub_grade_E4', 'sub_grade_E5', 'sub_grade_F1', 'sub_grade_F2',
       'sub_grade_F3', 'sub_grade_F4', 'sub_grade_F5', 'sub_grade_G1',
       'sub_grade_G2', 'sub_grade_G3', 'sub_grade_G4', 'sub_grade_G5',
       'term_ 36 months', 'term_ 60 months', 'int_rate_bin_High (16-20%)',
       'int_rate_bin_Low (8-12%)', 'int_rate_bin_Medium (12-16%)',
       'int_rate_bin_Very High (>20%)', 'int_rate_bin_Very Low (<8%)',
       'dti_bin_High (30-40)', 'dti_bin_Low (10-20)', 'dti_bin_Medium (20-30)',


# 2. Association Rule Mining Construction

In [5]:
frequent_itemsets = apriori(df, min_support=0.05, use_colnames=True, low_memory=True)

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=2)

rules = rules.sort_values('confidence', ascending=False)

c:\Users\Tohpati\anaconda3\envs\MachineLearning\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


`min_support = 0.0` <br>
Persentase minimum kemunculan kombinasi item dalam seluruh dataset. Nilai 0.05 berarti sebuah pola harus muncul setidaknya di 5% dari total transaksi pinjaman.

Alasan Pemilihan:
- Menghindari Outlier: Jika nilai terlalu rendah (misal 0.001), kita akan mendapatkan ribuan aturan yang hanya terjadi pada segelintir nasabah, yang tidak cukup kuat untuk dijadikan dasar kebijakan bank.
- Representasi Populasi: Angka 5% memastikan bahwa insight yang kita ambil mewakili kelompok nasabah yang cukup besar (massal) sehingga aksi bisnis yang diambil nantinya memiliki dampak yang signifikan secara volume


`metric = "lift"` <br>
Metrik yang mengukur seberapa besar kecenderungan kemunculan Antecedent dan Consequent secara bersamaan dibandingkan jika keduanya muncul secara acak (independen).

Alasan Pemilihan:
- Lift adalah metrik "kejujuran" dalam asosiasi. Berbeda dengan Confidence yang bisa menipu jika sebuah item memang sangat populer, Lift menunjukkan apakah ada hubungan sebab-akibat atau korelasi yang kuat antar variabel.

`min_threshold = 2` <br>
Batasan nilai Lift. Nilai Lift > 1 menunjukkan korelasi positif.

Alasan Pemilihan:
Angka 2 dipilih untuk mendapatkan aturan yang kuat secara statistik. Artinya, kemunculan pola tersebut 2 kali lebih sering terjadi dibandingkan jika variabel-variabelnya muncul secara kebetulan. Ini membantu kita menyaring hanya hubungan yang benar-benar "berarti" dan bukan sekadar kebetulan statistik.

# 3. Rule Evaluation & Metrics Formatting

### Initial Statistical Rules Discovery

In [6]:
import pandas as pd
pd.set_option('display.max_rows', None)

import pandas as pd

def pretty_rules(df_input):
    df = df_input.copy()
    
    # 1. Konversi frozenset ke string yang rapi tanpa tanda kurung
    df['antecedents'] = df['antecedents'].apply(lambda x: ', '.join(list(x)))
    df['consequents'] = df['consequents'].apply(lambda x: ', '.join(list(x)))
    
    # 2. Bulatkan angka metrik
    df['support'] = df['support'].round(3)
    df['confidence'] = df['confidence'].round(3)
    df['lift'] = df['lift'].round(3)
    
    # 3. Pilih kolom dengan urutan yang enak dibaca (Sisi kiri dan kanan terpisah)
    display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
    result = df[display_cols]
    
    return result

hasil_rapi = pretty_rules(rules)

print(f"Total rules yang ditemukan: {len(hasil_rapi)}")

# hasil_rapi.to_csv("results_apriori_accepted_loan.csv")


Total rules yang ditemukan: 1212


In [7]:
pd.reset_option('display.max_rows')

# 4. Rule Post-Processing & Optimization

### 4.1. Trivial Knowledge Filtering (Logical Redundancy)
membuang aturan yang secara sistematis pasti benar. Contohnya: sub_grade_B2 $\rightarrow$ grade_B. Karena secara definisi B2 adalah bagian dari Grade B, informasi ini tidak memberikan insight

In [8]:
def filter_trivial_rules(rules_df):
    def is_trivial(row):
        ant = list(row['antecedents'])
        cons = list(row['consequents'])
        
        for a in ant:
            # Cek jika antecedent adalah sub_grade (misal: sub_grade_B2) 
            # dan consequent adalah grade yang sama (misal: grade_B)
            if 'sub_grade_' in a:
                grade_from_sub = a.split('_')[-1][0] # Ambil huruf pertama setelah sub_grade_
                for c in cons:
                    if c == f'grade_{grade_from_sub}':
                        return True
        return False

    # Terapkan filter
    rules_non_trivial = rules_df[~rules_df.apply(is_trivial, axis=1)].copy()
    return rules_non_trivial

rules_filtered = filter_trivial_rules(rules)
print(f"Rules setelah menghapus trivial: {len(rules_filtered)}")

Rules setelah menghapus trivial: 1188


### 4.2 Superset Pruning (Structural Redundancy)
menghapus aturan yang "terlalu spesifik" tanpa memberikan peningkatan akurasi (confidence)

Jika aturan `Skor_FICO_Tinggi --> Bunga_Rendah` punya confidence 80%, <br>
maka aturan `Skor_FICO_Tinggi, Punya_Mobil --> Bunga_Rendah` yang juga punya confidence 80% akan dihapus. 

Menambahkan "Punya Mobil" tidak memberikan informasi baru yang mengubah hasil.

In [9]:
def remove_redundant_rules(rules_df):
    # Urutkan berdasarkan jumlah antecedent (terpendek dulu) dan confidence (tertinggi dulu)
    rules_df["ant_len"] = rules_df["antecedents"].apply(len)
    rules_df = rules_df.sort_values(by=["ant_len", "confidence"], ascending=[True, False])
    
    indices_to_drop = []
    
    # Iterasi untuk membandingkan setiap aturan
    for i, row_a in rules_df.iterrows():
        for j, row_b in rules_df.iterrows():
            if i == j: continue
            
            # Jika Consequent-nya sama
            if row_a['consequents'] == row_b['consequents']:
                # Jika antecedents A adalah superset dari antecedents B (A lebih spesifik dari B)
                # DAN confidence A tidak lebih baik secara signifikan dari B
                if row_b['antecedents'].issubset(row_a['antecedents']):
                    if row_a['confidence'] <= row_b['confidence']:
                        indices_to_drop.append(i)
                        break
                        
    return rules_df.drop(index=indices_to_drop).drop(columns=["ant_len"])

rules_final = remove_redundant_rules(rules_filtered)
print(f"Rules setelah menghapus redundansi: {len(rules_final)}")

Rules setelah menghapus redundansi: 877


### 4.3 Final Optimized Knowledge Base


In [11]:
hasil_akhir_rapi = pretty_rules(rules_final)

# Urutkan berdasarkan lift tertinggi untuk melihat insight paling berharga
hasil_akhir_rapi = hasil_akhir_rapi.sort_values('lift', ascending=False)

# Simpan hasil pembersihan ke CSV baru
hasil_akhir_rapi.to_csv("results_apriori_cleaned.csv", index=False)

print("Pembersihan selesai! File 'results_apriori_cleaned.csv' telah dibuat.")
hasil_akhir_rapi

Pembersihan selesai! File 'results_apriori_cleaned.csv' telah dibuat.


,antecedents,consequents,support,confidence,lift
1173,"fico_range_low_bin_High, int_rate_bin_Very Low...","last_fico_range_low_bin_High, grade_A, home_ow...",0.051,0.434,6.314
1158,"last_fico_range_low_bin_High, grade_A, home_ow...","fico_range_low_bin_High, int_rate_bin_Very Low...",0.051,0.748,6.314
1157,"grade_A, home_ownership_MORTGAGE, fico_range_l...","last_fico_range_low_bin_High, int_rate_bin_Ver...",0.051,0.678,6.289
1174,"last_fico_range_low_bin_High, int_rate_bin_Ver...","grade_A, home_ownership_MORTGAGE, fico_range_l...",0.051,0.478,6.289
1168,"grade_A, fico_range_low_bin_High","last_fico_range_low_bin_High, home_ownership_M...",0.051,0.392,6.287
...,...,...,...,...,...
103,grade_A,"last_fico_range_low_bin_High, dti_bin_Low (10-20)",0.055,0.286,2.023
1072,"last_fico_range_low_bin_High, home_ownership_M...","term_ 36 months, grade_A, int_rate_bin_Very Lo...",0.058,0.324,2.018
964,"last_fico_range_low_bin_High, home_ownership_M...","term_ 36 months, int_rate_bin_Very Low (<8%)",0.058,0.324,2.016
119,grade_A,"last_fico_range_low_bin_High, home_ownership_M...",0.069,0.359,2.011


# 5. Strategic Business Insights & Executive Summary

### Segment A (Kluster 2): Prime & Low-Risk Borrowers Profile

#### 1. Hubungan Tenor Pendek dengan Kualitas Kredit

rule: `term_ 36 months, fico_range_low_bin_High, int_rate_bin_Very Low (<8%) -> grade_A` <br>
klaster 2: Prime & Low-Risk Borrowers <br>

##### Insight Bisnis: 
nasabah yang memiliki skor FICO tinggi dan memilih bunga sangat rendah dengan jangka waktu cicilan pendek (3 tahun) dipastikan 100% masuk ke dalam kategori Grade A.

##### Aksi Bisnis:
Dorong nasabah yanng berkualitas tinggi untuk mengambil tenor 36 bulan dengan insentif bunga rendah, karena ini mempercepat peputaran modal bank dengan risiko mminim

#### 2. Stabilitas FICO pada Pemilik Rumah (Mortgage)

rule: `grade_A, home_ownership_MORTGAGE, fico_range_low_bin_High -> last_fico_range_low_bin_High, int_rate_bin_Very Low (<8%)` <br>
klaster 2: Prime & Low-Risk Borrowers <br>

##### Insight Bisnis:
nasabah grade A yang memiliki rumah (status mortgage) dan skor fico yang awalnya tinggi, cenderung berhasil mempertahankan skor kreditnya tetap tinggi hingga akhir periode pengamatan dan tetap menikmati bunga rendah, ini merupakan indikator perilaku keuangan yang stabil. Nasabah morgage lebih disiplin dalam menjaga skor mereka tetap tinggi

##### Aksi Bisnis:
Dalam strategi pemasaran, targetkan segmen pemilik rumah (MORTGAGE) karena mereka memiliki probabilitas lebih tinggi untuk menjadi nasabah berkualitas Grade A.

#### 3. Karakteristik Nasabah dengan Hutang Sehat (DTI)

rule: `fico_range_low_bin_High, dti_bin_Low (10-20), int_rate_bin_Very Low (<8%) -> grade_A` <br>
klaster 2: Prime & Low-Risk Borrowers <br>
baris: 172 <br>
support: 0.053 <br>
confidence: 1.0 <br>
lift: 5.218 <br>

##### Insight Bisnis:
nasabah dengan rasio hutang terhadap pendepatan yang ideal 10-20% disertai skor fico tinggi, secara konsisten akan diklarifikasikan ke grade A

##### Aksi Bisnis:
radio DTI di angka 10-20% menjadi zona aman bagi analisis kredit. Kelompok ini memiliki ruang finansial yang cukup untuk membayaran cicilan baru tanpa terbebani hutang lama, sehingga probabilitas persetujuan otomatis bisa ditingkatkan pada segmen ini


#### 4. Karakteristik Penyewa Rumah (RENT) yang Disiplin

rule: `home_ownership_RENT, int_rate_bin_Very Low (<8%) -> term_ 36 months, grade_A` <br>
klaster 2: Prime & Low-Risk Borrowers <br>
support: 0.052 <br>
confidence: 0.969 <br>
lift: 5.352 <br>

##### Insight Bisnis:
nasabah yang berstatus kontrak/sewa rumah namun berhasil mendapatkan bunga sangat rendah hampir selalu mengambil jangka waktu pendek (3 tahun) dan tergolong nasabah kualitas terbaik.

#### Aksi Bisnis:
Jangan meremehkan nasabah penyewa rumah. Jika mereka memilih tenor pendek (36 bulan), mereka adalah profil yang sangat disiplin. Bank dapat memberikan persetujuan cepat bagi penyewa rumah asalkan mereka mengambil tenor pendek.

#### 5. Kekuatan Riwayat Pelunasan (Fully Paid)
rule: `int_rate_bin_Very Low (<8%), loan_status_Fully Paid -> last_fico_range_low_bin_High, term_ 36 months, grade_A` <br>
klaster 2: Prime & Low-Risk Borrowers <br>
support: 0.055 <br>
confidence: 0.637 <br>
lift: 5.701 <br>

##### Insight Bisnis:
nasabah yang sudah pernah melunasi pinjaman sebelumnya dengan bunga rendah cenderung menjaga skor kreditnya tetap tinggi hingga saat ini.

##### Aksi Bisnis:
jadikan riwayat "Fully Paid" menjadi tiket emas, bagi nasabah yang ingin melakukan pinjaman berulang dengan limit yang lebih besar, karena loyalitas dan integrasi pembayarannya sudah teruji


#### 6. Konsistensi Nasabah "Lancar" (Current)

rule: `last_fico_range_low_bin_High, loan_status_Current -> home_ownership_MORTGAGE` <br>
klaster 2: Prime & Low-Risk Borrowers <br>
support:

##### Insight Bisnis:
Nasabah yang pembayaran cicilannya saat ini lancar dan memiliki skor fico kredit tinggi saat ini mayoritas adalah mereka yang memiliki cicilan rumah (Mortgage)

##### Aksi Bisnis:
Hal ini memperkuat temuan bahwa kepemilikan aset rumah (status mortgage) adalah prediktor terbaik untuk kelancaran pembayaran jangka panjang. Gunakan variabel ini sebagai bobot tinggi dalam sistem skoring otomatis.


#### 12. Kebutuhan Dana Besar bagi Nasabah Berkualitas
rule: `term_ 36 months, installment_bin_High, grade_A -> int_rate_bin_Very Low (<8%)` <br>
klaster 2: Prime & Low-Risk Borrowers <br>

Nasabah Grade A yang mengambil pinjaman dengan cicilan bulanan besar (nilai pinjaman tinggi) pada tenor 3 tahun tetap berhak mendapatkan bunga terendah.

##### Insight Bisnis:
Jangan ragu memberikan limit besar kepada nasabah Grade A. Data menunjukkan bahwa meskipun cicilannya tinggi (Installment High), profil risiko mereka tetap konsisten di zona bunga terendah. Ini adalah peluang untuk meningkatkan outstanding pinjaman tanpa menaikkan rasio NPL.

### Segment B (Kluster 1): Moderate-Risk & Growth Opportunities

#### 7. Profil Sukses Nasabah Grade B (Bunga Rendah)
rule: `int_rate_bin_Low (8-12%), loan_status_Fully Paid -> grade_B, term_ 36 months, home_ownership_MORTGAGE` <br>
klaster 1: Moderate-Risk & Small-Scale Borrowers <br>
support:

##### Insight Bisnis:
nasabah yang mendapatkan bunga rendah (8-12%) dan sudah melunasi pinjamannya cenderung merupakan pemilik rumah yang mengambil tenor 3 tahun dan berada di Grade B.

##### Aksi Bisnis:
ini adalah profil nasabah "Aspirasional". Mereka bukan yang paling elit, tapi sangat patuh membayar. Mereka adalah target pasar yang sangat stabil untuk produk-produk keuangan menengah-atas


#### 8. Dominasi Nasabah Grade B dalam Protofolio Aktif

rule: `loan_status_Current, int_rate_bin_Low (8-12%) -> grade_B` <br>
klaster 1: Moderate-Risk & Small-Scale Borrowers <br>
support:

##### Insight Bisnis:
hampir 90% nasabah yang statusnya lancar dan membayar bunga rendah (8-12%) adalah nasabah Grade B.

##### Aksi Bisnis:
kelompok ini adalah pemberi income pendapatan bunga tertinggi (lebih tinggi dari Grade A), namun tetap memiliki kedisiplinan bayar yang hampir setara. Menjaga kepuasan nasabah di segmen ini sangat krusial agar portofoio tetap menguntungkan

#### 9. Konsistensi Pembayaran pada Bunga Menengah
rule: `int_rate_bin_Medium (12-16%), grade_C -> loan_status_Current` <br>
klaster 1: Moderate-Risk & Small-Scale Borrowers <br>

##### Insight Bisnis:
nasabah Grade C yang mendapatkan bunga 12-16% secara konsisten menunjukkan status pembayaran yang lancar (Current).

Aturan ini memvalidasi efektivitas skoring kredit untuk Grace C. meskipun mereka bukan nasabah terbaik, pemberian bunga di level 12-16% tidak menyebabkan mereka gagal bayar (default)

#### 10. Ketahanan Nasabah dengan Skor Kredit Menengah
rule: `int_rate_bin_Low (8-12%), fico_range_low_bin_Medium -> grade_B, term_ 36 months` <br>
klaster 1: Moderate-Risk & Small-Scale Borrowers <br>

Nasabah dengan skor FICO menengah yang mengambil bunga rendah cenderung memilih tenor pendek (3 tahun) dan masuk dalam kategori Grade B.

##### Insight Bisnis:
Skor FICO "Medium" tidak selalu berarti risiko tinggi. Jika nasabah tersebut mengambil tenor pendek, mereka menunjukkan niat bayar yang kuat. Bank dapat menawarkan produk tenor pendek secara agresif kepada segmen skor menengah ini

#### 11. Keamanan pada Pinjaman Cicilan Kecil
rule: `installment_bin_Low, int_rate_bin_Very Low (<8%) -> term_ 36 months, grade_A` <br>
klaster 1: Moderate-Risk & Small-Scale Borrowers <br>

Pinjaman dengan nilai cicilan bulanan yang kecil dan bunga sangat rendah hampir selalu (Confidence 96%) berasal dari nasabah Grade A dengan tenor 3 tahun.

##### Insight Bisnis:
Pinjaman mikro atau kecil dengan bunga rendah sangat efektif untuk menarik nasabah berkualitas tinggi (Grade A). Ini bisa digunakan sebagai strategi "pintu masuk" untuk mendapatkan nasabah baru yang nantinya bisa ditawarkan produk asuransi atau investasi.

### Segment C (Kluster 0): High-Risk Warning Signals & Loss Prevention

#### 13. Sinyal Bahaya: Nasabah Gagal Bayar (Charged Off)
rule: `loan_status_Charged Off -> home_ownership_RENT, last_fico_range_low_bin_Low` <br>
klaster 0: High-Risk & High-Debt Borrowers <br>

Nasabah yang mengalami gagal bayar (Charged Off) memiliki kaitan sangat erat dengan status tempat tinggal sewa (RENT) dan penurunan skor FICO ke level terendah (Low) di akhir periode.

##### Insight Bisnis:
Status "Sewa Rumah" yang dikombinasikan dengan penurunan skor kredit adalah indikator paling kritis. Jika nasabah sewa rumah mulai menunjukkan penurunan skor FICO dalam pemantauan bulanan, bank harus segera melakukan tindakan penagihan proaktif sebelum terjadi gagal bayar.

#### 14. Risiko pada Nasabah Grade D (High Interest)
rule: `int_rate_bin_High (16-20%) -> grade_D` <br>
klaster 0: High-Risk & High-Debt Borrowers <br>

Nasabah yang dibebankan bunga tinggi (16-20%) hampir seluruhnya terkonsentrasi di Grade D.

##### Insight Bisnis:
Meskipun bunga tinggi memberikan margin keuntungan yang besar, kelompok ini adalah titik panas risiko. Rasio gagal bayar di Grade D biasanya jauh lebih tinggi. Bank perlu membatasi total eksposur (plafon pinjaman) pada segmen ini agar jika terjadi gagal bayar, dampaknya tidak mengguncang portofolio keseluruhan.

#### 15. Fenomena Penurunan Skor Kredit (FICO Downgrade)
rule: `fico_range_low_bin_High, loan_status_Charged Off -> last_fico_range_low_bin_Low` <br>
klaster 0: High-Risk & High-Debt Borrowers <br>

Bahkan nasabah yang awalnya memiliki skor FICO tinggi, jika mereka sampai pada status gagal bayar, skor mereka akan terjun bebas ke level terendah.

##### Insight Bisnis:
Tidak ada nasabah yang benar-benar "kebal" risiko. Aturan ini memperingatkan bahwa histori masa lalu yang baik (fico_range_low_bin_High) tidak menjamin keamanan jika kondisi ekonomi nasabah berubah. Pemantauan skor kredit secara berkala (behavioral scoring) jauh lebih penting daripada hanya mengandalkan skor saat pendaftaran (application scoring).

#### 16. Risiko Tenor Panjang (60 Bulan) di Grade Menengah
rule: `term_ 60 months, int_rate_bin_Medium (12-16%) -> grade_C` <br>
klaster 0: High-Risk & High-Debt Borrowers <br>

Pinjaman jangka panjang (5 tahun) dengan bunga menengah adalah karakteristik utama dari nasabah Grade C.

##### Insight Bisnis:
Semakin lama tenor pinjaman, semakin besar ketidakpastian yang dihadapi nasabah (risiko kehilangan pekerjaan, kesehatan, dll). Nasabah Grade C yang mengambil tenor 60 bulan harus dipantau lebih ketat karena mereka terpapar risiko ekonomi dalam jangka waktu yang lebih lama dibandingkan nasabah tenor 36 bulan.

#### 17. Risiko Tersembunyi pada Nasabah "Rent" (Sewa)
#### Memperkuat Aturan 13
rule: `home_ownership_RENT, last_fico_range_low_bin_Low -> loan_status_Charged Off` <br>
klaster 0: High-Risk & High-Debt Borrowers <br>

Nasabah yang menyewa tempat tinggal dan mengalami penurunan skor kredit hingga ke level terendah memiliki kecenderungan sangat kuat untuk berakhir pada status gagal bayar (Charged Off).

##### Insight Bisnis:
Status RENT seringkali menunjukkan mobilitas yang tinggi dan aset tetap yang minim. Kombinasi antara ketiadaan aset rumah dan skor kredit yang memburuk adalah sinyal terkuat bagi departemen penagihan untuk segera melakukan intervensi.